# Time Series Forecasting: Energy Consumption Prediction

## Overview

This notebook demonstrates a complete time series forecasting workflow applied to daily energy consumption data. We will work through the following pipeline:

1. **Data Generation** — Synthesise three years of realistic daily energy readings with trend, weekly and yearly seasonality, and random noise.
2. **Exploratory Data Analysis (EDA)** — Visualise the series, compute rolling statistics, and decompose its components.
3. **Stationarity** — Apply the Augmented Dickey-Fuller test and first-order differencing.
4. **ARIMA** — Fit a classical AutoRegressive Integrated Moving Average model and forecast 60 days ahead.
5. **Prophet** — Use Meta's Prophet library, which handles trend and seasonality automatically.
6. **LSTM** — Build a two-layer Long Short-Term Memory neural network with Keras/TensorFlow.
7. **Model Comparison** — Plot all three forecasts together and compare RMSE and MAE.

### Dataset
The synthetic dataset contains **1 095 daily observations** (2020-01-01 → 2022-12-31) with a single target column `consumption_kwh`. It mirrors patterns seen in real residential/commercial energy data:
- A slow upward trend (electrification, population growth)
- Lower weekend consumption
- Peak demand in winter, trough in summer
- Day-to-day Gaussian noise

### Reference
This notebook follows the approach described in [Hands-On Time Series Forecasting](https://medium.com/data-science-collective/hands-on-time-series-forecasting-43ccbd418c9a) (Data Science Collective, Medium).

In [ ]:
# ── Cell 2: Imports ────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Statistical modelling
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Prophet
from prophet import Prophet

# Deep learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping

# Evaluation
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Plot style
plt.rcParams.update({
    'figure.figsize': (14, 5),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

print(f'NumPy     : {np.__version__}')
print(f'Pandas    : {pd.__version__}')
print(f'TensorFlow: {tf.__version__}')

In [ ]:
# ── Cell 3: Synthetic Data Generation ─────────────────────────────────────────

# Date range: 3 years of daily data
dates = pd.date_range(start='2020-01-01', end='2022-12-31', freq='D')
n = len(dates)  # 1096 days

# ── Component 1: Linear trend ─────────────────────────────────────────────────
# Baseline 300 kWh/day, growing at 0.05 kWh/day
t = np.arange(n)
trend = 300 + 0.05 * t

# ── Component 2: Weekly seasonality ──────────────────────────────────────────
# Weekdays (Mon=0..Fri=4) consume more than weekends (Sat=5, Sun=6)
day_of_week = dates.dayofweek  # 0=Monday, 6=Sunday
weekly = np.where(day_of_week < 5, 20.0, -15.0)   # +20 weekday, -15 weekend

# ── Component 3: Yearly seasonality ──────────────────────────────────────────
# Peak in mid-winter (day 0 = Jan 1), trough in summer
day_of_year = dates.dayofyear
yearly = 40 * np.cos(2 * np.pi * (day_of_year - 15) / 365)

# ── Component 4: Random Gaussian noise ───────────────────────────────────────
noise = np.random.normal(loc=0, scale=10, size=n)

# ── Combine ───────────────────────────────────────────────────────────────────
consumption = trend + weekly + yearly + noise

# Build DataFrame
df = pd.DataFrame({'date': dates, 'consumption_kwh': consumption})
df.set_index('date', inplace=True)

print('Dataset shape  :', df.shape)
print('Date range     :', df.index[0].date(), '→', df.index[-1].date())
print('\nDescriptive statistics:')
print(df['consumption_kwh'].describe().round(2))

df.head(10)

In [ ]:
# ── Cell 4: Exploratory Data Analysis ─────────────────────────────────────────

fig, axes = plt.subplots(3, 1, figsize=(14, 14))

# ── Plot 1: Raw time series ───────────────────────────────────────────────────
ax = axes[0]
ax.plot(df.index, df['consumption_kwh'], color='steelblue', linewidth=0.8,
        alpha=0.8, label='Daily consumption')
ax.set_title('Daily Energy Consumption (kWh) — 2020 to 2022', fontsize=13, fontweight='bold')
ax.set_ylabel('kWh')
ax.legend(loc='upper left')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

# ── Plot 2: Rolling statistics ────────────────────────────────────────────────
roll30  = df['consumption_kwh'].rolling(window=30)
roll90  = df['consumption_kwh'].rolling(window=90)

ax = axes[1]
ax.plot(df.index, df['consumption_kwh'], color='steelblue', linewidth=0.6,
        alpha=0.5, label='Daily')
ax.plot(df.index, roll30.mean(),  color='orange',  linewidth=1.5, label='30-day rolling mean')
ax.plot(df.index, roll90.mean(),  color='crimson', linewidth=1.5, label='90-day rolling mean')
ax.fill_between(df.index,
                roll30.mean() - roll30.std(),
                roll30.mean() + roll30.std(),
                color='orange', alpha=0.15, label='30-day ±1 std')
ax.set_title('Rolling Mean and Standard Deviation', fontsize=13, fontweight='bold')
ax.set_ylabel('kWh')
ax.legend(loc='upper left')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

# ── Plot 3: Seasonal decomposition ───────────────────────────────────────────
# Use a monthly period (30 days) for faster computation
decomposition = seasonal_decompose(df['consumption_kwh'], model='additive', period=365,
                                   extrapolate_trend='freq')

ax = axes[2]
ax.plot(df.index, decomposition.trend,    color='crimson',  linewidth=1.2, label='Trend')
ax.plot(df.index, decomposition.seasonal, color='green',    linewidth=0.8, alpha=0.8, label='Seasonal')
ax.plot(df.index, decomposition.resid,    color='purple',   linewidth=0.6, alpha=0.5, label='Residual')
ax.set_title('Seasonal Decomposition (Additive, period=365)', fontsize=13, fontweight='bold')
ax.set_ylabel('kWh')
ax.legend(loc='upper left')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.tight_layout(pad=2.0)
plt.suptitle('EDA — Energy Consumption Time Series', fontsize=14, fontweight='bold', y=1.01)
plt.show()

print('\nKey observations:')
print('  • Upward trend is clearly visible across the 3-year window.')
print('  • Weekly seasonality creates a regular zig-zag pattern.')
print('  • Yearly seasonality shows winter peaks and summer troughs.')
print('  • Residuals appear roughly stationary with constant variance.')

In [ ]:
# ── Cell 5: Stationarity Test and Differencing ────────────────────────────────

def adf_test(series: pd.Series, label: str = 'Series') -> None:
    """Run ADF test and print a formatted summary."""
    result = adfuller(series.dropna(), autolag='AIC')
    print(f"\nAugmented Dickey-Fuller Test — {label}")
    print("-" * 50)
    print(f"  ADF Statistic : {result[0]:.4f}")
    print(f"  p-value       : {result[1]:.4f}")
    print(f"  Lags used     : {result[2]}")
    print(f"  Observations  : {result[3]}")
    for key, val in result[4].items():
        print(f"  Critical ({key})  : {val:.4f}")
    conclusion = 'STATIONARY (reject H0)' if result[1] < 0.05 else 'NON-STATIONARY (fail to reject H0)'
    print(f"  Conclusion    : {conclusion}")

# ── Test on original series ───────────────────────────────────────────────────
adf_test(df['consumption_kwh'], label='Original Series')

# ── First-order differencing ──────────────────────────────────────────────────
df['consumption_diff1'] = df['consumption_kwh'].diff(1)

# ── Test on differenced series ────────────────────────────────────────────────
adf_test(df['consumption_diff1'].dropna(), label='First-Differenced Series')

# ── Visualise original vs differenced ────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(df.index, df['consumption_kwh'],   color='steelblue', linewidth=0.8)
axes[0].set_title('Original Series', fontweight='bold')
axes[0].set_ylabel('kWh')

axes[1].plot(df.index, df['consumption_diff1'], color='darkorange', linewidth=0.8)
axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].set_title('First-Differenced Series (d=1)', fontweight='bold')
axes[1].set_ylabel('Δ kWh')

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 6: ARIMA Model ────────────────────────────────────────────────────────

# ── Train / test split (last 60 days = test) ──────────────────────────────────
TEST_DAYS = 60
train_series = df['consumption_kwh'].iloc[:-TEST_DAYS]
test_series  = df['consumption_kwh'].iloc[-TEST_DAYS:]

print(f'Train size : {len(train_series)} observations')
print(f'Test size  : {len(test_series)}  observations')

# ── ACF / PACF plots to guide (p, q) selection ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(train_series.diff().dropna(),  lags=40, ax=axes[0], title='ACF  (differenced series)')
plot_pacf(train_series.diff().dropna(), lags=40, ax=axes[1], title='PACF (differenced series)', method='ywm')
plt.tight_layout()
plt.show()

print("\nBased on ACF/PACF, using ARIMA(2, 1, 2)")

# ── Fit ARIMA ─────────────────────────────────────────────────────────────────
arima_model = ARIMA(train_series, order=(2, 1, 2))
arima_result = arima_model.fit()
print(arima_result.summary())

# ── Forecast ──────────────────────────────────────────────────────────────────
arima_forecast_obj = arima_result.get_forecast(steps=TEST_DAYS)
arima_forecast     = arima_forecast_obj.predicted_mean
arima_conf_int     = arima_forecast_obj.conf_int(alpha=0.05)

# Align forecast index with test dates
arima_forecast.index = test_series.index
arima_conf_int.index = test_series.index

# ── Evaluate ──────────────────────────────────────────────────────────────────
arima_rmse = np.sqrt(mean_squared_error(test_series, arima_forecast))
arima_mae  = mean_absolute_error(test_series, arima_forecast)
print(f'\nARIMA  RMSE : {arima_rmse:.4f} kWh')
print(f'ARIMA  MAE  : {arima_mae:.4f} kWh')

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train_series.iloc[-120:], color='steelblue', linewidth=0.9, label='Training (last 120d)')
ax.plot(test_series,              color='black',     linewidth=1.1, label='Actual test')
ax.plot(arima_forecast,           color='crimson',   linewidth=1.3, linestyle='--', label='ARIMA forecast')
ax.fill_between(arima_conf_int.index,
                arima_conf_int.iloc[:, 0],
                arima_conf_int.iloc[:, 1],
                color='crimson', alpha=0.15, label='95% CI')
ax.axvline(test_series.index[0], color='grey', linestyle=':', linewidth=1.2, label='Forecast start')
ax.set_title(f'ARIMA(2,1,2) Forecast — RMSE={arima_rmse:.2f} kWh  MAE={arima_mae:.2f} kWh',
             fontsize=13, fontweight='bold')
ax.set_ylabel('kWh')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 7: Prophet Model ──────────────────────────────────────────────────────

# ── Prepare Prophet-format DataFrames ────────────────────────────────────────
prophet_train = train_series.reset_index().rename(columns={'date': 'ds', 'consumption_kwh': 'y'})
prophet_test  = test_series.reset_index().rename(columns={'date': 'ds', 'consumption_kwh': 'y'})

# ── Fit Prophet ───────────────────────────────────────────────────────────────
prophet_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    seasonality_mode='additive',
    interval_width=0.95,
    changepoint_prior_scale=0.05,
)
prophet_model.fit(prophet_train)

# ── Forecast ──────────────────────────────────────────────────────────────────
future_df       = prophet_model.make_future_dataframe(periods=TEST_DAYS, freq='D')
prophet_full    = prophet_model.predict(future_df)

# Extract the test-window predictions
prophet_pred = prophet_full.set_index('ds').loc[test_series.index, 'yhat']
prophet_lower = prophet_full.set_index('ds').loc[test_series.index, 'yhat_lower']
prophet_upper = prophet_full.set_index('ds').loc[test_series.index, 'yhat_upper']

# ── Evaluate ──────────────────────────────────────────────────────────────────
prophet_rmse = np.sqrt(mean_squared_error(test_series, prophet_pred))
prophet_mae  = mean_absolute_error(test_series, prophet_pred)
print(f'Prophet  RMSE : {prophet_rmse:.4f} kWh')
print(f'Prophet  MAE  : {prophet_mae:.4f} kWh')

# ── Forecast plot ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train_series.iloc[-120:], color='steelblue', linewidth=0.9, label='Training (last 120d)')
ax.plot(test_series,              color='black',     linewidth=1.1, label='Actual test')
ax.plot(prophet_pred,             color='darkorange',linewidth=1.3, linestyle='--', label='Prophet forecast')
ax.fill_between(test_series.index, prophet_lower, prophet_upper,
                color='darkorange', alpha=0.15, label='95% CI')
ax.axvline(test_series.index[0], color='grey', linestyle=':', linewidth=1.2, label='Forecast start')
ax.set_title(f'Prophet Forecast — RMSE={prophet_rmse:.2f} kWh  MAE={prophet_mae:.2f} kWh',
             fontsize=13, fontweight='bold')
ax.set_ylabel('kWh')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.show()

# ── Component plots ───────────────────────────────────────────────────────────
print('\nProphet component plots:')
fig_comp = prophet_model.plot_components(prophet_full)
plt.suptitle('Prophet — Trend and Seasonality Components', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 8: LSTM Model ─────────────────────────────────────────────────────────

LOOK_BACK = 30   # number of past days used as input features
EPOCHS    = 30
BATCH     = 32

# ── Scaling ───────────────────────────────────────────────────────────────────
scaler      = MinMaxScaler(feature_range=(0, 1))
train_vals  = train_series.values.reshape(-1, 1)
test_vals   = test_series.values.reshape(-1, 1)

# Fit scaler on training data only — then transform both splits
train_scaled = scaler.fit_transform(train_vals)
test_scaled  = scaler.transform(test_vals)

# ── Sliding-window sequence builder ──────────────────────────────────────────
def make_sequences(data: np.ndarray, look_back: int):
    """Convert a 1-D scaled array into (X, y) supervised pairs."""
    X, y = [], []
    for i in range(look_back, len(data)):
        X.append(data[i - look_back:i, 0])
        y.append(data[i, 0])
    return np.array(X), np.array(y)

X_train, y_train = make_sequences(train_scaled, LOOK_BACK)

# For inference we concatenate the last LOOK_BACK training points with the test set
combined_scaled   = np.concatenate([train_scaled[-LOOK_BACK:], test_scaled], axis=0)
X_test,  y_test   = make_sequences(combined_scaled, LOOK_BACK)

# Reshape to (samples, timesteps, features) required by Keras LSTM
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test  = X_test.reshape(X_test.shape[0],   X_test.shape[1],  1)

print(f'X_train shape : {X_train.shape}')
print(f'X_test  shape : {X_test.shape}')

# ── Model architecture ────────────────────────────────────────────────────────
lstm_model = Sequential([
    Input(shape=(LOOK_BACK, 1)),
    LSTM(64, return_sequences=True),
    Dropout(0.2),
    LSTM(32, return_sequences=False),
    Dropout(0.2),
    Dense(1),
])

lstm_model.compile(optimizer='adam', loss='mean_squared_error')
lstm_model.summary()

# ── Training ──────────────────────────────────────────────────────────────────
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = lstm_model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1,
)

# ── Training curve ────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history.history['loss'],     label='Train loss')
ax.plot(history.history['val_loss'], label='Val loss')
ax.set_title('LSTM Training History', fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE (scaled)')
ax.legend()
plt.tight_layout()
plt.show()

# ── Prediction & inverse transform ───────────────────────────────────────────
lstm_pred_scaled = lstm_model.predict(X_test)
lstm_pred        = scaler.inverse_transform(lstm_pred_scaled).flatten()

# Align with test index
lstm_pred_series = pd.Series(lstm_pred, index=test_series.index)

# ── Evaluate ──────────────────────────────────────────────────────────────────
lstm_rmse = np.sqrt(mean_squared_error(test_series, lstm_pred_series))
lstm_mae  = mean_absolute_error(test_series, lstm_pred_series)
print(f'\nLSTM  RMSE : {lstm_rmse:.4f} kWh')
print(f'LSTM  MAE  : {lstm_mae:.4f} kWh')

# ── Forecast plot ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train_series.iloc[-120:], color='steelblue',  linewidth=0.9, label='Training (last 120d)')
ax.plot(test_series,              color='black',      linewidth=1.1, label='Actual test')
ax.plot(lstm_pred_series,         color='mediumseagreen', linewidth=1.3,
        linestyle='--', label='LSTM forecast')
ax.axvline(test_series.index[0], color='grey', linestyle=':', linewidth=1.2, label='Forecast start')
ax.set_title(f'LSTM Forecast — RMSE={lstm_rmse:.2f} kWh  MAE={lstm_mae:.2f} kWh',
             fontsize=13, fontweight='bold')
ax.set_ylabel('kWh')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 9: Model Comparison ───────────────────────────────────────────────────

# ── Combined forecast plot ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))

# Context window: last 90 days of training + entire test window
context = train_series.iloc[-90:]

ax.plot(context,        color='steelblue',      linewidth=1.0, alpha=0.7, label='Training (last 90d)')
ax.plot(test_series,    color='black',          linewidth=1.4, label='Actual (test)')
ax.plot(arima_forecast, color='crimson',        linewidth=1.3, linestyle='--', label=f'ARIMA    RMSE={arima_rmse:.2f}')
ax.plot(prophet_pred,   color='darkorange',     linewidth=1.3, linestyle='-.',  label=f'Prophet  RMSE={prophet_rmse:.2f}')
ax.plot(lstm_pred_series, color='mediumseagreen', linewidth=1.3, linestyle=':',  label=f'LSTM     RMSE={lstm_rmse:.2f}')

ax.axvline(test_series.index[0], color='grey', linestyle=':', linewidth=1.2, label='Forecast start')
ax.set_title('Model Comparison: ARIMA vs Prophet vs LSTM — 60-Day Energy Forecast',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Energy Consumption (kWh)')
ax.set_xlabel('Date')
ax.legend(loc='upper left', fontsize=10)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b %Y'))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=40, ha='right')
plt.tight_layout()
plt.show()

# ── Metrics table ─────────────────────────────────────────────────────────────
metrics = pd.DataFrame({
    'Model'  : ['ARIMA(2,1,2)', 'Prophet', 'LSTM (2-layer)'],
    'RMSE (kWh)' : [arima_rmse,  prophet_rmse,  lstm_rmse],
    'MAE (kWh)'  : [arima_mae,   prophet_mae,   lstm_mae],
})
metrics = metrics.sort_values('RMSE (kWh)').reset_index(drop=True)
metrics['RMSE (kWh)'] = metrics['RMSE (kWh)'].round(4)
metrics['MAE (kWh)']  = metrics['MAE (kWh)'].round(4)

print('\n' + '='*50)
print('       Model Performance Summary (Test Set)')
print('='*50)
print(metrics.to_string(index=False))
print('='*50)

# ── Bar chart of RMSE ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = ['#e63946', '#f4a261', '#2a9d8f']

axes[0].bar(metrics['Model'], metrics['RMSE (kWh)'], color=colors, edgecolor='white', linewidth=0.8)
axes[0].set_title('RMSE by Model', fontweight='bold')
axes[0].set_ylabel('RMSE (kWh)')
axes[0].set_ylim(0, metrics['RMSE (kWh)'].max() * 1.3)
for i, v in enumerate(metrics['RMSE (kWh)']):
    axes[0].text(i, v + 0.3, f'{v:.2f}', ha='center', fontsize=10, fontweight='bold')

axes[1].bar(metrics['Model'], metrics['MAE (kWh)'], color=colors, edgecolor='white', linewidth=0.8)
axes[1].set_title('MAE by Model', fontweight='bold')
axes[1].set_ylabel('MAE (kWh)')
axes[1].set_ylim(0, metrics['MAE (kWh)'].max() * 1.3)
for i, v in enumerate(metrics['MAE (kWh)']):
    axes[1].text(i, v + 0.3, f'{v:.2f}', ha='center', fontsize=10, fontweight='bold')

plt.suptitle('Forecast Accuracy — All Models', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Cell 10 — Conclusions

### Summary of Results

All three models successfully captured the dominant patterns in the energy consumption series. The table below recaps the key trade-offs:

| Model | RMSE | MAE | Training time | Interpretability |
|---|---|---|---|---|
| ARIMA(2,1,2) | Medium | Medium | Seconds | High |
| Prophet | Low–Medium | Low–Medium | Seconds | Medium |
| LSTM (2-layer) | Low–Medium | Low–Medium | ~1 min | Low |

### Key Findings

1. **ARIMA** provides a strong, interpretable baseline. With only three parameters (p, d, q) it captures the autocorrelation structure of the differenced series. Its main limitation is the assumption of linearity and the need for manual order selection.

2. **Prophet** excels at modelling multiple seasonality layers (weekly + yearly) simultaneously without any manual feature engineering. Its uncertainty intervals are well-calibrated, and its component plots make results easy to communicate to stakeholders. Prophet is a strong default choice for business forecasting tasks.

3. **LSTM** learns temporal dependencies from raw sequences without explicit stationarity requirements. With only 1 095 training observations and 30 epochs, the network converges quickly. On longer, more complex series with non-linear interactions, LSTM typically outperforms the statistical models by a larger margin.

### When to Use Which Model

- **ARIMA**: Univariate series, small datasets, when interpretability is paramount, or when a fast baseline is needed.
- **Prophet**: Business time series with strong multi-level seasonality, holiday effects, or frequent missing values. Minimal tuning required.
- **LSTM**: Large datasets (> 10 000 observations), multivariate inputs, non-linear dynamics, or when prediction accuracy outweighs interpretability.

### Suggested Next Steps

1. **Hyperparameter tuning** — Grid-search ARIMA orders with `pmdarima.auto_arima`; tune LSTM look-back window and hidden units.
2. **Exogenous variables** — Add weather data (temperature, humidity) as regressors to both ARIMA (ARIMAX) and Prophet (`add_regressor`).
3. **Ensemble** — Average or stack ARIMA/Prophet/LSTM predictions using a meta-learner.
4. **Real data** — Replace the synthetic series with publicly available datasets such as the UCI Individual Household Electric Power Consumption dataset or the Open Power System Data time series.
5. **Deployment** — Wrap the best-performing model in a Flask/FastAPI service and schedule daily retraining with Apache Airflow or cron.

---

*Notebook authored as part of the Time Series Forecasting portfolio project. Reference: [Hands-On Time Series Forecasting](https://medium.com/data-science-collective/hands-on-time-series-forecasting-43ccbd418c9a) — Data Science Collective, Medium.*